# Fashion Retrieval — Evaluation (Config C)

Evaluates retrieval performance for **Config C**: fine-tuned CLIP + BLIP-2 captions.

Loads fine-tuned CLIP weights and Config C FAISS indexes produced by the fine-tuning notebook.

---

### Inputs
- `vr-yolo-bbox-cropped-images` — query crops + master_crops.csv
- `blip-captions-data` — gallery_captions.json
- clip finetuned dataset — full_clip_finetuned.pt + idx_C_b07.bin + idx_C_b05.bin + item_index_map.csv

## 1. Install Packages

In [1]:
!pip uninstall -y faiss faiss-gpu
!pip install ftfy regex transformers faiss-cpu --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00


## 2. Imports

In [2]:
import os
import json
import numpy as np
import pandas as pd
import torch
import faiss
from PIL import Image
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import warnings
warnings.filterwarnings('ignore')

GPU = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Runtime device : {GPU}')
print('Imports complete!')

Runtime device : cuda
Imports complete!


## 3. Configuration

In [3]:
# ============================================================
#  SET YOUR TEAM ROLL NUMBERS AS SEEDS
# ============================================================
ROLL_SEEDS = [25, 29, 513, 521]   # replace with actual roll numbers
# ============================================================

BBOX_CROPS_DIR   = '/kaggle/input/datasets/akibatra25/vr-yolo-bbox-cropped-images'
CAPTIONS_DIR     = '/kaggle/input/datasets/akibatra25/blip-captions-data'
FINETUNED_DIR    = '/kaggle/input/datasets/akibatra25/clip-c-output'  # update after saving
INDEXES_AB_DIR   = '/kaggle/input/datasets/akibatra25/clip-ab-output'  # for item_index_map

K_LIST     = [5, 10, 15]
ENC_BATCH  = 64
CLIP_CKPT  = 'openai/clip-vit-base-patch32'

EVAL_CONFIGS = [
    ('Config_C_beta0.7', 'idx_C_b07.bin', 0.7),
    ('Config_C_beta0.5', 'idx_C_b05.bin', 0.5),
]

for tag, p in [('BBOX_CROPS_DIR', BBOX_CROPS_DIR), ('FINETUNED_DIR', FINETUNED_DIR)]:
    ok = 'Found ✓' if os.path.exists(p) else 'NOT FOUND ✗'
    print(f'[{ok}] {tag}')

print(f'\nRoll seeds : {ROLL_SEEDS}')

[Found ✓] BBOX_CROPS_DIR
[Found ✓] FINETUNED_DIR

Roll seeds : [25, 29, 513, 521]


## 4. Load Query Data and Gallery Metadata

In [4]:
full_table = pd.read_csv(os.path.join(BBOX_CROPS_DIR, 'master_crops.csv'))
qry_table  = full_table[full_table['split'] == 'query'].reset_index(drop=True)

def translate_path(saved_path):
    if pd.isna(saved_path): return saved_path
    for pfx in ['/kaggle/working/', '/kaggle/input/']:
        if saved_path.startswith(pfx):
            tail = saved_path.replace(pfx, '')
            for ds in ['vr-yolo-bbox-cropped-images/', 'datasets/akibatra25/vr-yolo-bbox-cropped-images/']:
                tail = tail.replace(ds, '')
            return os.path.join(BBOX_CROPS_DIR, tail)
    return saved_path

qry_table['img_path'] = qry_table['crop_path'].apply(translate_path)
qry_table['on_disk']  = qry_table['img_path'].apply(
    lambda p: os.path.exists(p) if isinstance(p, str) else False
)

if qry_table['on_disk'].sum() < len(qry_table) * 0.9:
    def direct_path(img_name):
        rel = img_name[4:] if img_name.startswith('img/') else img_name
        for sub in ['data/bbox_crops', 'data/yolo_crops']:
            p = os.path.join(BBOX_CROPS_DIR, sub, rel)
            if os.path.exists(p): return p
        return os.path.join(BBOX_CROPS_DIR, 'data/bbox_crops', rel)
    qry_table['img_path'] = qry_table['image_name'].apply(direct_path)
    qry_table['on_disk']  = qry_table['img_path'].apply(os.path.exists)

valid_qry = qry_table[qry_table['on_disk']].reset_index(drop=True)
item_map  = pd.read_csv(os.path.join(INDEXES_AB_DIR, 'item_index_map.csv'))

print(f'Valid query images : {len(valid_qry):,}')
print(f'Gallery map rows   : {len(item_map):,}')

Valid query images : 14,218
Gallery map rows   : 12,612


## 5. Load Fine-Tuned CLIP and Encode Queries

In [5]:
print(f'Loading CLIP base: {CLIP_CKPT}')
clip_proc = CLIPProcessor.from_pretrained(CLIP_CKPT)
clip_net  = CLIPModel.from_pretrained(CLIP_CKPT).to(GPU)

ft_path = os.path.join(FINETUNED_DIR, 'full_clip_finetuned.pt')
print(f'Loading fine-tuned weights: {ft_path}')
clip_net.load_state_dict(torch.load(ft_path, map_location=GPU))
print('Fine-tuned weights loaded ✓')

for p in clip_net.parameters():
    p.requires_grad = False
clip_net.eval()
VEC_DIM = clip_net.config.projection_dim
print(f'Vector dimension: {VEC_DIM}')


def get_image_vec(inp_dict):
    vis_out   = clip_net.vision_model(pixel_values=inp_dict['pixel_values'])
    projected = clip_net.visual_projection(vis_out.pooler_output)
    return projected / projected.norm(dim=-1, keepdim=True)


n_qry    = len(valid_qry)
qry_vecs = np.zeros((n_qry, VEC_DIM), dtype=np.float32)

print(f'Encoding {n_qry:,} query images...')
for s in tqdm(range(0, n_qry, ENC_BATCH), desc='Query encoding'):
    chunk = valid_qry.iloc[s : s + ENC_BATCH]
    imgs, ok_idx = [], []
    for i, (_, row) in enumerate(chunk.iterrows()):
        try:
            imgs.append(Image.open(row['img_path']).convert('RGB'))
            ok_idx.append(i)
        except Exception: pass
    if not imgs: continue
    inp = clip_proc(images=imgs, return_tensors='pt', padding=True).to(GPU)
    with torch.no_grad():
        vecs = get_image_vec(inp)
    for li, gi in enumerate(ok_idx):
        qry_vecs[s + gi] = vecs[li].cpu().numpy()

print(f'Query vectors shape: {qry_vecs.shape}')

Loading CLIP base: openai/clip-vit-base-patch32


preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading fine-tuned weights: /kaggle/input/datasets/akibatra25/clip-c-output/full_clip_finetuned.pt
Fine-tuned weights loaded ✓
Vector dimension: 512
Encoding 14,218 query images...


Query encoding: 100%|██████████| 223/223 [02:55<00:00,  1.27it/s]

Query vectors shape: (14218, 512)


## 6. Metric Functions

In [6]:
def compute_recall(retrieved, q_id, k):
    return int(any(r == q_id for r in retrieved[:k]))


def compute_ndcg(retrieved, q_id, n_relevant, k):
    dcg = 0.0
    for rank, rid in enumerate(retrieved[:k], start=1):
        if rid == q_id:
            dcg += 1.0 / np.log2(rank + 1)
    ideal_hits = min(n_relevant, k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0.0


def compute_ap(retrieved, q_id, n_relevant, k):
    hits = 0
    running_prec = 0.0
    for rank, rid in enumerate(retrieved[:k], start=1):
        if rid == q_id:
            hits += 1
            running_prec += hits / rank
    denom = min(n_relevant, k)
    return running_prec / denom if denom > 0 else 0.0


print('Metric functions ready ✓')

Metric functions ready ✓


## 7. Run Evaluation

In [7]:
gal_item_ids    = item_map['item_id'].tolist()
gal_item_counts = item_map['item_id'].value_counts().to_dict()
result_rows = []

for cfg_name, idx_file, beta in EVAL_CONFIGS:
    print(f'\n=== {cfg_name} ===')

    idx_path = os.path.join(FINETUNED_DIR, idx_file)
    srch_idx = faiss.read_index(idx_path)
    print(f'  Index: {srch_idx.ntotal:,} vectors')

    per_seed = {k: {'recall': [], 'ndcg': [], 'ap': []} for k in K_LIST}

    for seed in ROLL_SEEDS:
        np.random.seed(seed)
        torch.manual_seed(seed)

        n_samp = min(max(500, int(0.2 * len(valid_qry))), len(valid_qry))
        s_idx  = np.random.choice(len(valid_qry), n_samp, replace=False)
        s_vecs = qry_vecs[s_idx].astype(np.float32)
        s_ids  = valid_qry.iloc[s_idx]['item_id'].tolist()

        _, hits = srch_idx.search(s_vecs, 16)

        rec  = {k: [] for k in K_LIST}
        ndcg = {k: [] for k in K_LIST}
        aps  = {k: [] for k in K_LIST}

        for q_id, hit_row in zip(s_ids, hits):
            retrieved = [gal_item_ids[i] for i in hit_row if i < len(gal_item_ids)]
            n_rel     = gal_item_counts.get(q_id, 1)
            for k in K_LIST:
                rec[k].append(compute_recall(retrieved, q_id, k))
                ndcg[k].append(compute_ndcg(retrieved, q_id, n_rel, k))
                aps[k].append(compute_ap(retrieved, q_id, n_rel, k))

        for k in K_LIST:
            per_seed[k]['recall'].append(np.mean(rec[k]))
            per_seed[k]['ndcg'].append(np.mean(ndcg[k]))
            per_seed[k]['ap'].append(np.mean(aps[k]))

        print(f'  Seed {seed}: R@10={np.mean(rec[10]):.4f}  NDCG@10={np.mean(ndcg[10]):.4f}  mAP@10={np.mean(aps[10]):.4f}')

    for k in K_LIST:
        result_rows.append({
            'config': cfg_name, 'K': k,
            'Recall_mean': np.mean(per_seed[k]['recall']),
            'Recall_std' : np.std(per_seed[k]['recall']),
            'NDCG_mean'  : np.mean(per_seed[k]['ndcg']),
            'NDCG_std'   : np.std(per_seed[k]['ndcg']),
            'mAP_mean'   : np.mean(per_seed[k]['ap']),
            'mAP_std'    : np.std(per_seed[k]['ap']),
        })

print('\nEvaluation complete!')


=== Config_C_beta0.7 ===
  Index: 12,612 vectors
  Seed 25: R@10=0.9216  NDCG@10=0.6518  mAP@10=0.5577
  Seed 29: R@10=0.9226  NDCG@10=0.6484  mAP@10=0.5533
  Seed 513: R@10=0.9184  NDCG@10=0.6461  mAP@10=0.5498
  Seed 521: R@10=0.9198  NDCG@10=0.6458  mAP@10=0.5492

=== Config_C_beta0.5 ===
  Index: 12,612 vectors
  Seed 25: R@10=0.9216  NDCG@10=0.6518  mAP@10=0.5577
  Seed 29: R@10=0.9226  NDCG@10=0.6484  mAP@10=0.5533
  Seed 513: R@10=0.9184  NDCG@10=0.6461  mAP@10=0.5498
  Seed 521: R@10=0.9198  NDCG@10=0.6458  mAP@10=0.5492

Evaluation complete!


## 8. Results Table

In [8]:
results_df = pd.DataFrame(result_rows)

print('\n=== CONFIG C EVALUATION RESULTS ===')
print(f'Seeds: {ROLL_SEEDS}  |  Format: mean ± std')
print()

for cfg_name, _, _ in EVAL_CONFIGS:
    rows = results_df[results_df['config'] == cfg_name]
    print(f'Config: {cfg_name}')
    print(f'  {"K":>4}  {"Recall@K":>14}  {"NDCG@K":>14}  {"mAP@K":>14}')
    print(f'  {"-"*52}')
    for _, r in rows.iterrows():
        print(f'  K={int(r["K"]):>2}  '
              f'{r["Recall_mean"]:.4f}±{r["Recall_std"]:.4f}  '
              f'{r["NDCG_mean"]:.4f}±{r["NDCG_std"]:.4f}  '
              f'{r["mAP_mean"]:.4f}±{r["mAP_std"]:.4f}')
    print()

results_df.to_csv('/kaggle/working/eval_results_C.csv', index=False)
print('Results saved to eval_results_C.csv')


=== CONFIG C EVALUATION RESULTS ===
Seeds: [25, 29, 513, 521]  |  Format: mean ± std

Config: Config_C_beta0.7
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.8799±0.0032  0.6341±0.0032  0.5567±0.0039
  K=10  0.9206±0.0016  0.6480±0.0024  0.5525±0.0034
  K=15  0.9377±0.0007  0.6624±0.0022  0.5582±0.0033

Config: Config_C_beta0.5
     K        Recall@K          NDCG@K           mAP@K
  ----------------------------------------------------
  K= 5  0.8799±0.0032  0.6341±0.0032  0.5567±0.0039
  K=10  0.9206±0.0016  0.6480±0.0024  0.5525±0.0034
  K=15  0.9377±0.0007  0.6624±0.0022  0.5582±0.0033

Results saved to eval_results_C.csv
